In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import gc
from ydata_synthetic.synthesizers.timeseries import TimeSeriesSynthesizer
from ydata_synthetic.synthesizers import ModelParameters, TrainParameters # <--- Added TrainParameters
import os
import joblib
import os
class DatasetConfig:
    def __init__(self, name, csv_path, window_size, sample_len, output_dir):
        self.name = name
        self.csv_path = csv_path
        self.window_size = window_size
        self.sample_len = sample_len
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

In [ ]:
class DoppelGANgerManager:
    def __init__(self, config, scaler_path):
        self.config = config
        self.model_dir = os.path.join(self.config.output_dir, f"{self.config.name}")
        os.makedirs(self.config.output_dir, exist_ok=True)

        self.df = pd.read_csv(self.config.csv_path)
        self.numeric_cols = self.df.columns.tolist()
        
        self.scaler = joblib.load(scaler_path)
        self.model = None

        
        seq_len = self.config.window_size
        remainder = len(self.df) % seq_len
        if remainder != 0:
            self.df = self.df.iloc[:-remainder]
        
        
    def train(self, epochs=400, batch_size=128):
        if os.path.exists(self.model_dir):
            self.model = TimeSeriesSynthesizer.load(self.model_dir)
            return

        model_args = ModelParameters(
            batch_size=batch_size,
            lr=0.001,
            betas=(0.2, 0.9),
            latent_dim=20,
            gp_lambda=2,
            pac=1
        )

        train_args = TrainParameters(
            epochs=epochs,
            sequence_length=self.config.window_size,
            sample_length=self.config.sample_len,
            rounds=1,
            measurement_cols=self.numeric_cols
        )

        self.model = TimeSeriesSynthesizer(
            modelname="doppelganger",
            model_parameters=model_args
        )

        self.model.fit(
            data=self.df,
            train_arguments=train_args,
            num_cols=self.numeric_cols,
            cat_cols=[]
        )

        if not os.path.exists(self.model_dir):
            os.makedirs(self.model_dir, exist_ok=True)

        model_dir_abs = os.path.abspath(self.model_dir)
        self.model.save(model_dir_abs)
        # print("    > Training complete.")


    def generate(self, num_samples=None):
        if num_samples == None:
            num_samples = len(self.df) // self.config.window_size

        if self.model is None:
            raise RuntimeError("Model not trained or loaded.")

        synth_data = self.model.sample(n_samples=num_samples)

        synth_df = pd.concat(synth_data, axis=0).reset_index(drop=True)


        synth_df[self.numeric_cols] = self.scaler.inverse_transform(
            synth_df[self.numeric_cols]
        )

        output_path = self.config.csv_path.replace(
            "numerical.csv",
            "doppelganger_synthetic.csv"
        )

        synth_df.round(2).to_csv(output_path, index=False)
        print(f"    > Saved synthetic data to {output_path}")

        # return synth_df



In [3]:

WINDOW_SIZE_DAILY = 30
SAMPLE_LEN_DAILY = 10  

WINDOW_SIZE_HOURLY = 48
SAMPLE_LEN_HOURLY = 12

WINDOW_SIZE_MINUTELY = 80
SAMPLE_LEN_MINUTELY = 20

# Google Stocks Data
cfg_stocks = DatasetConfig("GOOGL_Stocks", "./data/GOOGL_STOCKS/GOOGL_STOCKS_numerical.csv", WINDOW_SIZE_DAILY, SAMPLE_LEN_DAILY, "./doppelganger_models")
doppelganger_stocks = DoppelGANgerManager(cfg_stocks, './scaler/GOOGL_STOCKS_scaler.pkl')
doppelganger_stocks.train(epochs=500, batch_size=64)
doppelganger_stocks.generate()


tf.keras.backend.clear_session()
tf.compat.v1.reset_default_graph()
gc.collect()

# Occupancy Detection Data
cfg_occupancy = DatasetConfig("OccupancyDetection", "./data/OccupancyDetection/OccupancyDetection_numerical.csv", WINDOW_SIZE_MINUTELY, SAMPLE_LEN_MINUTELY, "./doppelganger_models")
doppelganger_occupancy = DoppelGANgerManager(cfg_occupancy, './scaler/OccupancyDetection_scaler.pkl')
doppelganger_occupancy.train(epochs=500, batch_size=64)
doppelganger_occupancy.generate()

tf.keras.backend.clear_session()
tf.compat.v1.reset_default_graph()
gc.collect()

# Power Consumption Data
cfg_power = DatasetConfig("PowerConsumption", "./data/PowerConsumption/PowerConsumption_numerical.csv", WINDOW_SIZE_MINUTELY, SAMPLE_LEN_MINUTELY, "./doppelganger_models")
doppelganger_power = DoppelGANgerManager(cfg_power, './scaler/PowerConsumption_scaler.pkl')
doppelganger_power.train(epochs=800, batch_size=64)
doppelganger_power.generate()

tf.keras.backend.clear_session()
tf.compat.v1.reset_default_graph()
gc.collect()

# Traffic Volume Data
cfg_traffic = DatasetConfig("TrafficVolume", "./data/TrafficVolume/TrafficVolume_numerical.csv", WINDOW_SIZE_HOURLY, SAMPLE_LEN_HOURLY, "./doppelganger_models")
doppelganger_traffic = DoppelGANgerManager(cfg_traffic, './scaler/TrafficVolume_scaler.pkl')
doppelganger_traffic.train(epochs=500, batch_size=64)
doppelganger_traffic.generate()

c:\Users\usuario\.conda\envs\ydatasynthetic_env\lib\site-packages\ydata_synthetic\synthesizers\timeseries\doppelganger\network.py:13: UserWarning: `tf.layers.dense` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Dense` instead.
  output = tf.compat.v1.layers.dense(


Instructions for updating:
Colocations handled automatically by placer.
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


c:\Users\usuario\.conda\envs\ydatasynthetic_env\lib\site-packages\ydata_synthetic\synthesizers\timeseries\doppelganger\network.py:274: UserWarning: `tf.nn.rnn_cell.LSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  cell = tf.compat.v1.nn.rnn_cell.LSTMCell(



INFO:tensorflow:Restoring parameters from c:\Users\usuario\Desktop\北航\数据挖掘\期末\doppelganger_models\GOOGL_Stocks\doppelganger
    > Saved synthetic data to ./data/GOOGL_STOCKS/GOOGL_STOCKS_doppelganger_synthetic.csv


c:\Users\usuario\.conda\envs\ydatasynthetic_env\lib\site-packages\ydata_synthetic\synthesizers\timeseries\doppelganger\network.py:13: UserWarning: `tf.layers.dense` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Dense` instead.
  output = tf.compat.v1.layers.dense(


...

c:\Users\usuario\.conda\envs\ydatasynthetic_env\lib\site-packages\ydata_synthetic\synthesizers\timeseries\doppelganger\network.py:274: UserWarning: `tf.nn.rnn_cell.LSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  cell = tf.compat.v1.nn.rnn_cell.LSTMCell(


INFO:tensorflow:Restoring parameters from c:\Users\usuario\Desktop\北航\数据挖掘\期末\doppelganger_models\PowerConsumption\doppelganger
    > Saved synthetic data to ./data/PowerConsumption/PowerConsumption_doppelganger_synthetic.csv


c:\Users\usuario\.conda\envs\ydatasynthetic_env\lib\site-packages\ydata_synthetic\synthesizers\timeseries\doppelganger\network.py:13: UserWarning: `tf.layers.dense` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Dense` instead.
  output = tf.compat.v1.layers.dense(


c:\Users\usuario\.conda\envs\ydatasynthetic_env\lib\site-packages\ydata_synthetic\synthesizers\timeseries\doppelganger\network.py:274: UserWarning: `tf.nn.rnn_cell.LSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  cell = tf.compat.v1.nn.rnn_cell.LSTMCell(


INFO:tensorflow:Restoring parameters from c:\Users\usuario\Desktop\北航\数据挖掘\期末\doppelganger_models\TrafficVolume\doppelganger
    > Saved synthetic data to ./data/TrafficVolume/TrafficVolume_doppelganger_synthetic.csv
